# 🚀 手动运行教程（satellite_ai）

本 Notebook 是 `satellite_ai` 改进代码的**手动运行教程**，一步步带你跑通「训练 → 评估」全流程。

- 前置：虚拟环境 `sssiiieee的文件夹/venv` 已建好、依赖已装好。
- 大部分步骤是在**终端**里执行的命令（复制粘贴即可），用 `bash` 代码块展示。
- 也有少量**可直接运行**的代码单元格，用来验证环境。


## 第 0 步：确认就绪（已帮你准备好）

- ✅ 虚拟环境：`sssiiieee的文件夹/venv`（torch / stable-baselines3 / gymnasium 等已装好）
- ✅ 改进代码：本文件夹下的 `sat_env.py` / `train.py` / `metrics.py` / `evaluate.py`


In [ ]:
# 验证虚拟环境是否配置正确（选择 venv 内核后运行本单元格）
import stable_baselines3, gymnasium, torch, numpy, matplotlib
print("stable-baselines3", stable_baselines3.__version__)
print("gymnasium", gymnasium.__version__)
print("torch", torch.__version__)
print("numpy", numpy.__version__)
print("environment OK")

## 第 1 步：打开终端并进入 satellite_ai 目录

在 VSCode 里打开 `satellite_ai` 文件夹，菜单 `终端 → 新建终端`，或手动 cd：

```bash
cd "D:\Projects\code\satelite\applications-satellite.attitude.control.system\sssiiieee的文件夹\satellite_ai"
```

> ⚠️ 重要：所有命令都要在 `satellite_ai` 目录下执行（脚本用相对路径输出结果）。


## 第 2 步：激活虚拟环境

按终端类型选一种：

```bash
# Git Bash（推荐）
source ../venv/Scripts/activate

# PowerShell
..\venv\Scripts\Activate.ps1

# CMD
..\venv\Scripts\activate.bat
```

激活成功后命令行前会出现 `(venv)`。

不想激活也行，之后所有 `python` 换成 `../venv/Scripts/python.exe` 即可。


## 第 3 步：跑训练

```bash
python single_axis/train.py sac     # SAC（推荐，默认）
# 或
python single_axis/train.py ppo     # PPO 对比
```

会发生什么：

1. 打印 `[ds] 开始训练 SAC，总步数 100000，种子 42`
2. 每 1 万步在确定性环境评估一次，自动保存**最优模型**到 `satellite_ai_best_model/`
3. 训练结束保存最终模型 `satellite_ai_best_model/sac_satellite_final.zip`

耗时：约 10–12 分钟（CPU）。

产物：

```
satellite_ai/
├── satellite_ai_tensorboard/   # 训练曲线（可用 tensorboard 查看）
└── satellite_ai_best_model/    # 最优模型 + 最终模型
```

> 💡 想快速试跑（几秒）：把 `train.py` 里的 `TOTAL_TIMESTEPS = 100_000` 临时改成 `5_000`，跑完再改回来。


## 第 4 步：跑评估 + 泛化测试

训练完成后：

```bash
python single_axis/evaluate.py
```

会输出：

1. 一张 `sac_ood_generalization.png`（不同初始角度下的收敛曲线）
2. 一张量化指标表（settle / overshoot / IAE / energy / success）

其中 `settle`（稳定时间）越小、`success` 越 True 越好。


## 第 5 步：（可选）打开讲解 Notebook

双击 `explanation.ipynb` → 右上角选 venv 内核 → 从上到下运行，边看讲解边跑演示。


## 第 6 步：（可选）看训练曲线

```bash
python -m tensorboard --logdir satellite_ai_tensorboard
```

浏览器打开 http://localhost:6006 查看 reward 曲线。


## 常见问题排查

| 现象 | 原因 / 解决 |
|------|------------|
| `ModuleNotFoundError: No module named 'sat_env'` | 没在 `satellite_ai` 目录运行，先 cd 进去 |
| `No module named 'stable_baselines3'` | 没激活 venv，先做第 2 步 |
| 训练曲线震荡 / 发散 | 正常，SAC 前期会震荡，看最终收敛效果 |
| 想改初始角度范围 / 干扰强度 | 改 `sat_env.py` 里的 `config` 字典 |


## 三轴模块（进阶）：训练三轴姿态控制

单轴只是 1 自由度双积分器，问题太简单。三轴才有耦合项 `ω×(Iω)` 和四元数运动学的非线性，
也是 RL 相对 PID 更能体现优势的地方。本模块动力学与 PID 三轴侧**完全一致**
（`max_torque=2.0 N·m`、`dt=0.01s`、惯量 `diag[1.2,1.0,0.8]`），保证对比公平。

```bash
# 训练三轴（SAC，约 25~30 分钟）
python three_axis/train3d.py sac

# 三轴评估 + PID vs RL 对比
python three_axis/evaluate3d.py three_axis/output/models_v2/sac_satellite3d_v2_final sac
```

产物（与单轴分开存放，不互相覆盖）：

```
satellite_ai/
├── three_axis/output/models_v2/       # 三轴模型（v2）
├── three_axis/output/tensorboard/     # 三轴训练曲线
└── three_axis/output/sac_vs_pid_3d.png   # 三轴对比图
```


### 三轴成果怎么读

`evaluate3d.py` 会输出一张 **PID vs SAC 对比表**（每个初始姿态各一行）。它把三轴姿态误差
折算成「等效转角」这一个标量，直接复用 `metrics.py` 的指标（稳定时间/超调/IAE/能耗/成功率）。

- 看「稳定时间」和「success」两列即可快速判断谁更好；
- **三轴同时大角度（如 45°/45°/45°）** 是 PID 容易失稳、RL 容易占优的场景，是重点对比对象。


## 模型辨识 + MPC 模块（补齐立项第②项）

立项书三大内容里唯一没做的是「模型辨识 + 预测控制」，本模块补齐它：

```bash
# 1. 系统辨识：最小二乘拟合离散模型 x_{k+1} = A x_k + B u_k
python mpc/system_id.py

# 2. 三方对比：PID vs MPC vs SAC
python mpc/compare.py
```

- `system_id.py` 打印辨识出的 A、B（应几乎等于真实双积分器），并保存到 `mpc/identified_model.npz`；
- `compare.py` 输出 PID / MPC / SAC 三方对比表 + 对比图 `mpc/pid_mpc_sac.png`。

**结果解读**：MPC 基于精确辨识的模型，速度/超调/稳态精度全面最优（能耗略高）；
SAC 最省能量但有 ~0.43° 残留；PID 简单但慢、超调大。


## 三轴姿态可视化动画（ds30）

```bash
python visualization/animate_attitude.py
```

可选参数：初始欧拉角（roll pitch yaw，度），默认 `0 0 60`（单轴偏航 60°），例如：

```bash
python visualization/animate_attitude.py 30 20 -40   # 自定义初始姿态
```

生成两个 GIF：`visualization/output/sac_attitude.gif` 和 `pid_attitude.gif`。
左图为卫星本体坐标系（红X/绿Y/蓝Z）随控制收敛旋转，右图为等效转角误差收敛曲线。
